# TenzorXAI Backend Training on Colab

Run these cells from top to bottom. This notebook generates the synthetic backend data, trains the LightGBM model, and downloads `backend_data.zip` for your local project.

## 1. Upload Project Zip

Upload a zip named like `TenzorXAI-colab.zip`. Recommended local Git Bash command before uploading:

```bash
cd /e/TenzorXAI
zip -r TenzorXAI-colab.zip backend README.md LICENSE -x "backend/data/*"
```

In [ ]:
from google.colab import files

uploaded = files.upload()
zip_name = next(iter(uploaded))
print(f"Uploaded: {zip_name}")

In [ ]:
!rm -rf /content/TenzorXAI
!mkdir -p /content/TenzorXAI
!unzip -q "$zip_name" -d /content/TenzorXAI

import os

project_root = "/content/TenzorXAI"

# If the zip contains one top-level folder, step into it automatically.
if not os.path.exists(os.path.join(project_root, "backend")):
    entries = [
        os.path.join(project_root, item)
        for item in os.listdir(project_root)
        if os.path.isdir(os.path.join(project_root, item))
    ]
    for entry in entries:
        if os.path.exists(os.path.join(entry, "backend")):
            project_root = entry
            break

assert os.path.exists(os.path.join(project_root, "backend", "requirements.txt")), "Could not find backend/requirements.txt after unzip. Check the uploaded zip structure."
print(f"Project root: {project_root}")
%cd $project_root

## 2. Install Backend Requirements

In [ ]:
!python --version
!python -m pip install --upgrade pip
!python -m pip install -r backend/requirements.txt

## 3. Generate Synthetic Data

This creates `synthetic_100k.parquet`, `locality_metadata.csv`, `circle_rates.csv`, and `locality_confidence.json` under `backend/data`.

In [ ]:
!python backend/scripts/synthetic_generator.py

## 4. Train LightGBM Model

This reads `backend/data/synthetic_100k.parquet` and writes `backend/data/model.pkl`.

In [ ]:
!python backend/scripts/train_model.py

## 5. Verify Outputs

In [ ]:
from pathlib import Path

required_outputs = [
    "backend/data/synthetic_100k.parquet",
    "backend/data/locality_metadata.csv",
    "backend/data/circle_rates.csv",
    "backend/data/locality_confidence.json",
    "backend/data/model.pkl",
]

for output in required_outputs:
    path = Path(output)
    assert path.exists(), f"Missing expected output: {output}"
    print(f"OK {output} ({path.stat().st_size / 1024 / 1024:.2f} MB)")

## 6. Download Generated Backend Data

After download, unzip `backend_data.zip` into your local `E:\TenzorXAI` project.

In [ ]:
!rm -f backend_data.zip
!zip -r backend_data.zip backend/data

from google.colab import files
files.download("backend_data.zip")

## 7. Local Run Command After Download

Back on your laptop, run the API only:

```bash
cd /e/TenzorXAI
source .venv/Scripts/activate
python -m uvicorn main:app --reload --app-dir backend
```

Health check: `http://127.0.0.1:8000/health`